In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/social-media-extremism-detection-challenge/sample_submission.csv
/kaggle/input/social-media-extremism-detection-challenge/train.csv
/kaggle/input/social-media-extremism-detection-challenge/test.csv


In [2]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# Load data
train = pd.read_csv("/kaggle/input/social-media-extremism-detection-challenge/train.csv")
test  = pd.read_csv("/kaggle/input/social-media-extremism-detection-challenge/test.csv")
sample = pd.read_csv("/kaggle/input/social-media-extremism-detection-challenge/sample_submission.csv")

# Text cleaning function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"\@\w+|\#","", text)
    text = re.sub(r"\d+","", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+"," ", text).strip()
    return text

# Apply cleaning
train["clean_message"] = train["Original_Message"].apply(clean_text)
test["clean_message"]  = test["Original_Message"].apply(clean_text)

# Map labels to numeric
label_map = {"NON_EXTREMIST": 0, "EXTREMIST": 1}
train["label"] = train["Extremism_Label"].map(label_map)

# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    train["clean_message"], train["label"], test_size=0.15, random_state=42, stratify=train["label"]
)

# Build pipeline
model = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1,3), max_features=30000)),
    ("clf", LinearSVC())
])

# Train model
model.fit(X_train, y_train)

# Validate
val_preds = model.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, val_preds))
print("\nClassification Report:\n", classification_report(y_val, val_preds))

# Predict on test
test_preds = model.predict(test["clean_message"])

# Convert numeric back to labels
reverse_map = {0: "NON_EXTREMIST", 1: "EXTREMIST"}
final_preds = [reverse_map[p] for p in test_preds]

# Prepare submission
submission = pd.DataFrame({
    "ID": test["ID"],
    "Extremism_Label": final_preds
})

# Ensure same order and length as test
submission = submission.sort_values("ID").reset_index(drop=True)
submission.to_csv("submission.csv", index=False)

print("\nSubmission file created: submission.csv")
submission.head()

Validation Accuracy: 0.8076923076923077

Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.80      0.80       165
           1       0.81      0.82      0.81       173

    accuracy                           0.81       338
   macro avg       0.81      0.81      0.81       338
weighted avg       0.81      0.81      0.81       338


Submission file created: submission.csv


,ID,Extremism_Label
0,2251,NON_EXTREMIST
1,2252,NON_EXTREMIST
2,2253,NON_EXTREMIST
3,2254,NON_EXTREMIST
4,2255,NON_EXTREMIST
